# Badge 3: Data Application Builders Workshop: Lesson 3: Moving Data from SiS to Snowflake

The col function, the multiselect, the ORDERS table and the order insert.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE3_DABW_ADMIN` (ACCOUNTADMIN), `BADGE3_DABW_ROLE` (SYSADMIN) and `BADGE3_DABW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE3_DABW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE3_DABW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE3_DABW_WH"      # stands in for the course's COMPUTE_WH
import ast, io
from snowflake.snowpark.context import get_active_session

CUSTOM = 'SMOOTHIES.PUBLIC."Custom Smoothie Order Form"'
PENDING = 'SMOOTHIES.PUBLIC."Pending Smoothie Orders"'


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result


def write_app(app, app_code):
    """SQL equivalent of editing the app in Snowsight's Streamlit editor: write streamlit_app.py into the app's live version."""
    ast.parse(app_code)  # the code must at least be valid Python
    path = f"snow://streamlit/{app}/versions/live/streamlit_app.py"
    get_active_session().file.put_stream(io.BytesIO(app_code.encode()), path, auto_compress=False, overwrite=True)
    print(f"wrote {len(app_code.splitlines())} lines to {path}")

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: ❄️ The Snowpark COL Function

> **Course:** Remove the SelectBox (screenshots 2026011415022, 023), import `col`, and bring back only the FRUIT_NAME column (025, 031).

Python, added to the app (not SQL):

```python
from snowflake.snowpark.functions import col
```

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
st.dataframe(data=my_dataframe, use_container_width=True)
''')

In [ ]:
%%sql -r run_app_3a
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

## Page: 🧰 The Streamlit Multi-select Widget

> **Course:** Add a Multiselect and comment out the line that shows the dataframe (screenshot 2026011415051).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )
''')

> **Course:** Display the LIST with `st.write()` and `st.text()` (screenshot 2026011415053).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

st.write(ingredients_list)
st.text(ingredients_list)
''')

In [ ]:
%%sql -r run_app_3b2
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

## Page: 🧾 Clean Up the GUI & Create an Orders Table

> **Course:** Clean Up Empty Brackets with an IF block (screenshots 2026011415063, 071).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    st.write(ingredients_list)
    st.text(ingredients_list)
''')

> **Course:** Create a Place to Store Order Data: a table in SMOOTHIES named ORDERS, owned by SYSADMIN, one text column INGREDIENTS with a 200-character limit (screenshot 2026011415074).

In [ ]:
%%sql -r create_orders
-- From screenshot 2026011415074
use role {{SYSADMIN_ROLE}};
create table SMOOTHIES.PUBLIC.ORDERS
(
ingredients varchar(200)
);

## Page: 🔀 Converting a LIST to a STRING

> **Course:** Create the INGREDIENTS_STRING variable inside the IF block (screenshot 2026011415094); add the FOR block (103); output the string (104).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    st.write(ingredients_list)
    st.text(ingredients_list)

    ingredients_string = ''
''')

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    st.write(ingredients_list)
    st.text(ingredients_list)

    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen
''')

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    st.write(ingredients_list)
    st.text(ingredients_list)

    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen

    st.write(ingredients_string)
''')

In [ ]:
%%sql -r run_app_3f
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

## Page: 💾 Storing the Orders in Snowflake

> **Course:** Build a SQL Insert Statement & Test It: delete the two write lines, add a space after each fruit name (screenshot 2026011415121) and add the course's `my_insert_stmt` (122).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen + ' '

    st.write(ingredients_string)


    my_insert_stmt = """ insert into smoothies.public.orders(ingredients)
            values ('""" + ingredients_string + """')"""

    st.write(my_insert_stmt)
''')

> **Course:** Copy the SQL statement and paste it in a worksheet to make sure it works (screenshots 2026011415122, 124).

In [ ]:
%%sql -r test_insert_statement
-- From screenshot 2026011415124: the statement the app printed
insert into smoothies.public.orders(ingredients)
values ('Cantaloupe Guava Jackfruit Elderberries Figs ');

In [ ]:
%%sql -r select_orders
select * from smoothies.public.orders;

> **Course:** Insert the Order into Snowflake (screenshot dabw086).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen + ' '

    #st.write(ingredients_string)


    my_insert_stmt = """ insert into smoothies.public.orders(ingredients)
            values ('""" + ingredients_string + """')"""

    #st.write(my_insert_stmt)
    if ingredients_string:
        session.sql(my_insert_stmt).collect()

        st.success('Your Smoothie is ordered!', icon="✅")
''')

## Page: 🚰 Fixing the Flow

> **Course:** Truncate the Orders Table (screenshot dabw089).

In [ ]:
%%sql -r truncate_orders
truncate table smoothies.public.orders;

> **Course:** Add a Submit Button (screenshot dabw090).

In [ ]:
write_app(CUSTOM, r'''# Import python packages
import streamlit as st
from snowflake.snowpark.context import get_active_session

# Write directly to the app
st.title(":cup_with_straw: Customize Your Smoothie! :cup_with_straw:")
st.write(
    """Choose the fruits you want in your custom Smoothie!
    """)

from snowflake.snowpark.functions import col

session = get_active_session()
my_dataframe = session.table("smoothies.public.fruit_options").select(col('FRUIT_NAME'))
#st.dataframe(data=my_dataframe, use_container_width=True)

ingredients_list = st.multiselect(
    'Choose up to 5 ingredients:'
    , my_dataframe
    )

if ingredients_list:
    ingredients_string = ''

    for fruit_chosen in ingredients_list:
        ingredients_string += fruit_chosen + ' '

    #st.write(ingredients_string)


    my_insert_stmt = """ insert into smoothies.public.orders(ingredients)
            values ('""" + ingredients_string + """')"""

    #st.write(my_insert_stmt)
    time_to_insert = st.button('Submit Order')

    if time_to_insert:
        session.sql(my_insert_stmt).collect()

        st.success('Your Smoothie is ordered!', icon="✅")
''')

In [ ]:
%%sql -r run_app_3i
-- SQL equivalent of clicking Run (a Streamlit app run shows up as an EXECUTE STREAMLIT query)
execute streamlit SMOOTHIES.PUBLIC."Custom Smoothie Order Form"();

> **Course:** Submit a Few More Orders: at least 5 DIFFERENT order rows in your table. Each insert below is what one click of Submit Order runs (the statement the app builds).

In [ ]:
%%sql -r submit_order_1
insert into smoothies.public.orders(ingredients) values ('Cantaloupe Ziziphus Jujube Tangerine ');

In [ ]:
%%sql -r submit_order_2
insert into smoothies.public.orders(ingredients) values ('Apples Guava Raspberries Yerba Mate Vanilla Fruit ');

In [ ]:
%%sql -r submit_order_3
insert into smoothies.public.orders(ingredients) values ('Elderberries Ximenia Ziziphus Jujube Tangerine ');

In [ ]:
%%sql -r submit_order_4
insert into smoothies.public.orders(ingredients) values ('Dragon Fruit Honeydew Guava Apples Kiwi ');

In [ ]:
%%sql -r submit_order_5
insert into smoothies.public.orders(ingredients) values ('Cantaloupe Guava Jackfruit Elderberries Figs ');

In [ ]:
%%sql -r check_orders
select * from smoothies.public.orders;

## Page: 🤖 DORA DABW002

### 🤖 DORA check DABW002 goes here
The course has the learner run **DABW002** (at least 5 different orders) at this point.
It is in `dora/dora_tests.ipynb`, section **DABW002**, as a local check plus the exact DORA code.